# OWID Panel Data for Triple Shield Hypothesis - Demo Notebook

## Overview

This notebook demonstrates the **OWID Panel Dataset for the Triple Shield Hypothesis** - a comprehensive panel dataset of 50 post-1990 democratizers (1990-2022).

The **Triple Shield Hypothesis** posits that democratic resilience is sustained by three interacting institutions:
1. **Education** (human capital development)
2. **Social Protection** (welfare state institutions)
3. **Inequality Reduction** (economic parity)

This dataset combines 8 OWID datasets plus World Bank API data to test whether welfare-state institutions mediate the link between inequality, education, and democratic quality.

### Dataset Contents

- **1,641 country-year observations** from 50 post-1990 democratizers
- **Variables**: V-Dem Electoral Democracy Index (outcome), Gini coefficient, mean years of schooling, social protection spending (% GDP), GDP per capita, natural resource rents, population, and derived interaction terms
- **Sources**: OWID (V-Dem, Gini, education, social spending, GDP, population, resource rents) and World Bank API

### What This Notebook Does

1. Loads a curated demo subset of the full dataset
2. Explores the data structure and missingness patterns
3. Computes descriptive statistics
4. Visualizes key relationships
5. Demonstrates basic regression analysis

In [ ]:
# Install dependencies - follows aii-colab pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru - NOT on Colab, always install
_pip('loguru==0.7.2')

# numpy, pandas, matplotlib - pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'seaborn==0.13.2', 'scipy==1.16.3')

print('Dependencies installed successfully!')

In [ ]:
# Imports
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# NumPy 2.0 compatibility shims (if needed)
if not hasattr(np, 'alltrue'): np.alltrue = np.all
if not hasattr(np, 'sometrue'): np.sometrue = np.any
if not hasattr(np, 'product'): np.product = np.prod

print('All imports successful!')

In [ ]:
# Data loading helper - GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-90d6bf-the-triple-shield-revisited-education-we/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL failed: {e}")
        pass
    
    # Local fallback
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading function defined.')

In [ ]:
# Load the demo data
data = load_data()

# Extract examples from the nested structure
dataset_name = data['datasets'][0]['dataset']
examples = data['datasets'][0]['examples']

print(f"Dataset: {dataset_name}")
print(f"Number of examples: {len(examples)}")
print(f"\nFirst example keys: {list(examples[0].keys())}")

## Data Exploration

Now let's parse the JSON input strings and convert the data into a pandas DataFrame for analysis.
Each example has:
- `input`: JSON string of features (gini, education_years, social_spending, etc.)
- `output`: V-Dem Electoral Democracy Index score (as string)
- `metadata_*`: Additional information about the observation

In [ ]:
# Parse the data into a pandas DataFrame
rows = []
for ex in examples:
    # Parse the input JSON string
    features = json.loads(ex['input'])
    
    # Create a row with features + metadata
    row = {
        **features,  # Unpack all features
        'vdem_electoral': float(ex['output']),
        'country': ex['metadata_country'],
        'year': ex['metadata_year'],
        'fold': ex['metadata_fold']
    }
    rows.append(row)

df = pd.DataFrame(rows)

print("DataFrame shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

In [ ]:
# Data types and missing values
print("Data types:")
print(df.dtypes)
print("\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0])
print("\nMissing values (%):")
print((df.isnull().sum() / len(df) * 100).round(1))

## Descriptive Statistics

Let's examine the distribution of key variables in the dataset.

In [ ]:
# Descriptive statistics for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
print("Descriptive Statistics:")
print(df[numeric_cols].describe().round(2))

# Country distribution
print("\nObservations per country:")
print(df['country'].value_counts().sort_index())

## Compute Interaction Terms

The Triple Shield Hypothesis involves interaction terms:
- **gini × education**: How inequality moderates the education effect
- **gini × social**: How inequality moderates the social spending effect  
- **triple interaction**: gini × education × social (all three together)

Let's compute these for rows where the component variables are available.

In [ ]:
# Compute interaction terms where possible
df['gini_x_education'] = df.apply(
    lambda row: row['gini'] * row['education_years'] if pd.notnull(row['gini']) and pd.notnull(row['education_years']) else None,
    axis=1
)

df['gini_x_social'] = df.apply(
    lambda row: row['gini'] * row['social_spending'] if pd.notnull(row['gini']) and pd.notnull(row['social_spending']) else None,
    axis=1
)

df['triple_interaction'] = df.apply(
    lambda row: row['gini'] * row['education_years'] * row['social_spending'] 
    if pd.notnull(row['gini']) and pd.notnull(row['education_years']) and pd.notnull(row['social_spending']) 
    else None,
    axis=1
)

print("Interaction terms computed.")
print(f"\nRows with gini_x_education: {df['gini_x_education'].notnull().sum()}")
print(f"Rows with gini_x_social: {df['gini_x_social'].notnull().sum()}")
print(f"Rows with triple_interaction: {df['triple_interaction'].notnull().sum()}")

## Visualizations

Let's visualize key relationships in the data to understand patterns related to the Triple Shield Hypothesis.

In [ ]:
# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create subplots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Histogram of V-Dem scores
axes[0, 0].hist(df['vdem_electoral'], bins=10, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('V-Dem Electoral Democracy Index')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Democratic Quality')

# Boxplot by fold
df.boxplot(column='vdem_electoral', by='fold', ax=axes[0, 1])
axes[0, 1].set_xlabel('Cross-Validation Fold')
axes[0, 1].set_ylabel('V-Dem Score')
axes[0, 1].set_title('V-Dem Scores by CV Fold')
axes[0, 1].get_figure().suptitle('')

# Education vs V-Dem
axes[0, 2].scatter(df['education_years'], df['vdem_electoral'], alpha=0.6)
axes[0, 2].set_xlabel('Mean Years of Schooling')
axes[0, 2].set_ylabel('V-Dem Score')
axes[0, 2].set_title('Education vs Democratic Quality')

# Gini vs V-Dem
gini_valid = df[df['gini'].notnull()]
if len(gini_valid) > 0:
    axes[1, 0].scatter(gini_valid['gini'], gini_valid['vdem_electoral'], alpha=0.6)
    axes[1, 0].set_xlabel('Gini Coefficient')
    axes[1, 0].set_ylabel('V-Dem Score')
    axes[1, 0].set_title('Inequality vs Democratic Quality')
else:
    axes[1, 0].text(0.5, 0.5, 'No valid Gini data', ha='center', va='center', transform=axes[1, 0].transAxes)

# GDP per capita vs V-Dem
axes[1, 1].scatter(np.log10(df['gdp_per_capita']), df['vdem_electoral'], alpha=0.6)
axes[1, 1].set_xlabel('Log10 GDP per Capita')
axes[1, 1].set_ylabel('V-Dem Score')
axes[1, 1].set_title('GDP per Capita vs Democratic Quality')

# Social spending vs V-Dem
social_valid = df[df['social_spending'].notnull()]
if len(social_valid) > 0:
    axes[1, 2].scatter(social_valid['social_spending'], social_valid['vdem_electoral'], alpha=0.6)
    axes[1, 2].set_xlabel('Social Protection Spending (% GDP)')
    axes[1, 2].set_ylabel('V-Dem Score')
    axes[1, 2].set_title('Social Spending vs Democratic Quality')
else:
    axes[1, 2].text(0.5, 0.5, 'No valid social spending data', ha='center', va='center', transform=axes[1, 2].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix (for available data)
corr_cols = ['gini', 'education_years', 'social_spending', 'gdp_per_capita', 
             'resource_rents', 'vdem_electoral']
corr_df = df[corr_cols].dropna()

if len(corr_df) > 1:
    corr_matrix = corr_df.corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Matrix of Key Variables')
    plt.tight_layout()
    plt.show()
    
    print("Correlations with V-Dem Electoral Democracy Index:")
    print(corr_matrix['vdem_electoral'].sort_values(ascending=False))
else:
    print("Not enough complete data to compute correlations.")

## Simple Regression Analysis

Let's run a simple linear regression to demonstrate how this dataset can be used for analysis.
We'll regress V-Dem scores on education, inequality (gini), and social spending.

In [ ]:
# Simple regression using statsmodels
try:
    import statsmodels.api as sm
    
    # Prepare data - use rows with no missing values in key variables
    reg_cols = ['education_years', 'gini', 'social_spending', 'gdp_per_capita']
    reg_df = df[reg_cols + ['vdem_electoral']].dropna()
    
    if len(reg_df) > 5:
        X = reg_df[reg_cols]
        X = sm.add_constant(X)  # Add intercept
        y = reg_df['vdem_electoral']
        
        # Fit model
        model = sm.OLS(y, X).fit()
        
        print("Regression Results:")
        print(model.summary())
    else:
        print(f"Not enough complete observations for regression (only {len(reg_df)} rows with complete data)")
        print("In the full dataset (1,641 observations), this analysis would be more robust.")
        
except ImportError:
    print("statsmodels not installed. Install with: pip install statsmodels")
    print("\nSimple correlation analysis instead:")
    for col in ['education_years', 'gini', 'social_spending']:
        valid = df[[col, 'vdem_electoral']].dropna()
        if len(valid) > 2:
            corr = valid[col].corr(valid['vdem_electoral'])
            print(f"Correlation of {col} with V-Dem: {corr:.3f} (n={len(valid)})")

## Summary

This demo notebook has demonstrated:

1. **Data Loading**: Successfully loaded the OWID panel dataset using the GitHub URL pattern with local fallback

2. **Data Structure**: The dataset contains country-year observations with features related to the Triple Shield Hypothesis

3. **Data Exploration**: Examined missing data patterns, descriptive statistics, and country coverage

4. **Interaction Terms**: Computed the key interaction terms (gini×education, gini×social, triple interaction)

5. **Visualizations**: Created plots showing relationships between inequality, education, social spending, and democratic quality

6. **Regression Analysis**: Demonstrated how to use the dataset for regression analysis

### Next Steps for Full Analysis

With the full dataset (1,641 observations), you could:
- Run panel data models (fixed effects, random effects)
- Test the triple interaction hypothesis formally
- Conduct robustness checks with alternative specifications
- Analyze subgroups (e.g., by region, by level of democracy)

### Data Availability

The demo dataset has some missing values (as does the full dataset):
- Gini coefficient: ~54% missing
- Education: ~34% missing  
- Social spending: ~52% missing

The full dataset uses multiple imputation and supplementary sources to address this.